### Circuit cutting

This example shows how to use OpenQARP's circuit cutting functionality to evaluate the expectation value of a Hamiltonian on a circuit that is too wide for a single device.

The key idea: a 4-qubit circuit is automatically partitioned into two 2-qubit subcircuits. Each gate at the partition boundary is decomposed into six single-qubit experiments via quasi-probability decomposition (QPD). The subcircuit results are recombined to recover the original expectation value.

The entry point is `CuttingPrimitive`, a standard `PrimitiveAlgorithm` that plugs into any OpenQARP engine — no pytket or external backend required.

#### 1. Build the circuit

In [ ]:
from qarp.blocks import CompositeBlock, HnBlock, LinearEntanglingBlock

n_qubits = 4
block = CompositeBlock(
    blocks=[
        HnBlock(n_qubits),
        LinearEntanglingBlock(n_qubits, circular=False, use_cz=False),
    ]
)
block.build()
block.plot(decompose_boxes=True)

#### 2. Define the Hamiltonian

In [ ]:
from qarp.operators import QubitOperator

hamiltonian = (
    + 0.16988452027940318  * QubitOperator("Z0")
    + 0.21886306781219608  * QubitOperator("Z0 Z1 Z2")
    + 0.4544288414432624  * QubitOperator("Y0 Y1 Y2")
    + 0.8841443262423423  * QubitOperator("X0 X1 Y2")
    + 0.4544288414432624  * QubitOperator("X0 X1 X2")
    + 0.6
)

#### 3. Set the capacity constraint

`max_subcircuit_qubits=2` tells the cutter that no subcircuit may exceed 2 qubits.
For a 4-qubit circuit this forces at least one cut, splitting the circuit into two 2-qubit pieces.

#### 4. Run the cutting primitive

In [ ]:
from qarp.algorithms import CuttingPrimitive
from qarp.engines import QarpEngine

n_shots = 10000

cutting_prim = CuttingPrimitive(
    ket=block,
    operator=hamiltonian,
    max_subcircuit_qubits=2,
    n_shots=n_shots,
)

engine = QarpEngine()
engine.build([cutting_prim])
result_cutting = engine.run()

print(f"Expectation value (circuit cutting): {result_cutting[0]:.4f}")

#### 5. Inspect the decomposition

After `build()` you can inspect how many cuts were made and how many sub-experiments were generated.

In [ ]:
qpd = cutting_prim._qpd

print(f"Number of cuts:              {qpd.n_cuts}")
n_experiments = len(qpd.coefficients)
print(f"Number of QPD experiments:   {n_experiments}  (= 6^{qpd.n_cuts})")
print(f"Number of subcircuits:       {qpd.n_subcircuits}")
print(f"Subcircuit qubit groups:     {cutting_prim._cutter_result.subcircuit_qubits}")
print(f"Total sub_blocks built:      {len(cutting_prim.sub_blocks)}")
print(f"Sampling overhead:           {qpd.overhead():.0f}×")

#### 6. Comparison: exact simulation without cutting

We verify the cutting result against `PauliAveraging`, which runs the full 4-qubit circuit without splitting.

In [ ]:
from qarp.algorithms import PauliAveraging

pauli_avg = PauliAveraging(ket=block, operator=hamiltonian, n_shots=50000)
engine.build([pauli_avg])
result_exact = engine.run()

print(f"Expectation value (PauliAveraging, no cut): {result_exact[0]:.4f}")
print(f"Expectation value (circuit cutting):        {result_cutting[0]:.4f}")
print(f"Absolute difference:                        {abs(result_exact[0] - result_cutting[0]):.4f}")

#### Notes

- The circuit cutting result converges to the exact value as `n_shots` grows, but the sampling overhead is `6^n_cuts` times larger than a direct measurement.
- The `force_max_number_cuts=True` flag disables the safety guard (default: warn if `n_cuts ≥ config.max_number_of_cuts = 6`).